#Worksheet - 4
#Building a Fully Connected Network (FCN) for Devnagari Digit Classification

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


##Objective

In this exercise, we build and train a Fully Connected Network (FCN) using TensorFlow/Keras to classify Devnagari handwritten digits.
The dataset is loaded manually using PIL, preprocessed into NumPy arrays, used to train the FCN, then evaluated, saved, loaded again, and used for prediction.

#Task 1: Data Preparation

In [2]:
import zipfile
import os

zip_path = "/content/drive/MyDrive/AI and Machine Learning/dataset/devnagari digit.zip"
extract_path = "/content/devnagari_digit_dataset"

# Create extraction folder
os.makedirs(extract_path, exist_ok=True)

# Unzip the file
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("Dataset extracted successfully.")
print("Extracted to:", extract_path)

Dataset extracted successfully.
Extracted to: /content/devnagari_digit_dataset


In [3]:
import os

for root, dirs, files in os.walk(extract_path):
    print("Folder:", root)
    print("Subfolders:", dirs[:10])
    print("Files:", files[:5])
    print("-" * 50)

Folder: /content/devnagari_digit_dataset
Subfolders: ['DevanagariHandwrittenDigitDataset']
Files: []
--------------------------------------------------
Folder: /content/devnagari_digit_dataset/DevanagariHandwrittenDigitDataset
Subfolders: ['Test', 'Train']
Files: []
--------------------------------------------------
Folder: /content/devnagari_digit_dataset/DevanagariHandwrittenDigitDataset/Test
Subfolders: ['digit_3', 'digit_8', 'digit_6', 'digit_2', 'digit_7', 'digit_0', 'digit_5', 'digit_9', 'digit_4', 'digit_1']
Files: []
--------------------------------------------------
Folder: /content/devnagari_digit_dataset/DevanagariHandwrittenDigitDataset/Test/digit_3
Subfolders: []
Files: ['29543.png', '40618.png', '77272.png', '71249.png', '5264.png']
--------------------------------------------------
Folder: /content/devnagari_digit_dataset/DevanagariHandwrittenDigitDataset/Test/digit_8
Subfolders: []
Files: ['56700.png', '42191.png', '42219.png', '69151.png', '71349.png']
----------------

## Loading the Data

In this task, we:
- load the Devnagari digit images from the folder structure,
- convert images to grayscale,
- resize them to 28×28,
- normalize pixel values to the range [0, 1],
- extract labels from folder names,
- and prepare the data for Keras.

Since the dataset is stored as a ZIP file in Google Drive, we first mount Google Drive and extract the dataset into the Colab working directory. After extraction, we inspect the folder structure and load the images.
Code

In [ ]:
# ============================
# Task 1: Data Preparation
# ============================

# Step 1: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Step 2: Import required libraries
import os
import zipfile
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.utils import to_categorical

# ============================
# Step 3: Automatically find ZIP file in Google Drive
# ============================

drive_base = "/content/drive/MyDrive"
zip_file_name_keywords = ["devnagari", "digit"]   # keywords to search in file name

def find_zip_file(base_path, keywords):
    matched_files = []

    for root, dirs, files in os.walk(base_path):
        for file in files:
            file_lower = file.lower()
            if file_lower.endswith(".zip") and all(k.lower() in file_lower for k in keywords):
                matched_files.append(os.path.join(root, file))

    return matched_files

matched_zip_files = find_zip_file(drive_base, zip_file_name_keywords)

if len(matched_zip_files) == 0:
    raise FileNotFoundError(
        "No matching ZIP file found in Google Drive.\n"
        "Please check your file name or upload the ZIP again."
    )
elif len(matched_zip_files) == 1:
    zip_path = matched_zip_files[0]
else:
    print("Multiple matching ZIP files found:")
    for i, path in enumerate(matched_zip_files):
        print(f"{i+1}. {path}")
    zip_path = matched_zip_files[0]
    print(f"\nUsing the first match: {zip_path}")

print("\nZIP file found at:")
print(zip_path)

# ============================
# Step 4: Define extraction folder
# ============================

extract_path = "/content/devnagari_digit_dataset"
os.makedirs(extract_path, exist_ok=True)

# ============================
# Step 5: Extract ZIP file
# ============================

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("\nDataset extracted successfully.")

# ============================
# Step 6: Automatically find Train and Test folders
# ============================

def find_train_test_dirs(base_path):
    train_dir = None
    test_dir = None

    for root, dirs, files in os.walk(base_path):
        for d in dirs:
            if d.lower() == "train":
                train_dir = os.path.join(root, d)
            elif d.lower() == "test":
                test_dir = os.path.join(root, d)

    return train_dir, test_dir

train_dir, test_dir = find_train_test_dirs(extract_path)

print("\nTrain directory:", train_dir)
print("Test directory:", test_dir)

if train_dir is None or test_dir is None:
    raise FileNotFoundError(
        "Train or Test folder not found inside extracted dataset.\n"
        "Please check your ZIP folder structure."
    )

# ============================
# Step 7: Load images from folder
# ============================

img_height, img_width = 28, 28

def load_images_from_folder(folder):
    images = []
    labels = []

    class_names = sorted([
        d for d in os.listdir(folder)
        if os.path.isdir(os.path.join(folder, d))
    ])

    class_map = {name: i for i, name in enumerate(class_names)}

    print(f"\nLoading from: {folder}")
    print("Classes found:", class_names)

    for class_name in class_names:
        class_path = os.path.join(folder, class_name)
        label = class_map[class_name]

        for filename in os.listdir(class_path):
            img_path = os.path.join(class_path, filename)

            try:
                img = Image.open(img_path).convert("L")         # grayscale
                img = img.resize((img_width, img_height))      # resize to 28x28
                img = np.array(img, dtype=np.float32) / 255.0  # normalize

                images.append(img)
                labels.append(label)

            except Exception as e:
                print(f"Skipping {img_path}: {e}")

    return np.array(images), np.array(labels), class_names

# ============================
# Step 8: Create training and testing arrays
# ============================

x_train, y_train, class_names = load_images_from_folder(train_dir)
x_test, y_test, _ = load_images_from_folder(test_dir)

print("\nOriginal shapes:")
print("x_train:", x_train.shape)
print("y_train:", y_train.shape)
print("x_test:", x_test.shape)
print("y_test:", y_test.shape)

# ============================
# Step 9: Reshape for Keras
# ============================

x_train = x_train.reshape(-1, 28, 28, 1)
x_test = x_test.reshape(-1, 28, 28, 1)

# ============================
# Step 10: One-hot encode labels
# ============================

num_classes = len(class_names)

y_train_cat = to_categorical(y_train, num_classes=num_classes)
y_test_cat = to_categorical(y_test, num_classes=num_classes)

print("\nAfter reshaping and encoding:")
print("x_train:", x_train.shape)
print("y_train_cat:", y_train_cat.shape)
print("x_test:", x_test.shape)
print("y_test_cat:", y_test_cat.shape)

# ============================
# Step 11: Show sample images
# ============================

plt.figure(figsize=(12, 6))

for i in range(min(10, len(x_train))):
    plt.subplot(2, 5, i + 1)
    plt.imshow(x_train[i].reshape(28, 28), cmap='gray')
    plt.title(f"Label: {y_train[i]}")
    plt.axis("off")

plt.tight_layout()
plt.show()

# ============================
# Step 12: Print class mapping
# ============================

print("\nClass Names:")
for idx, name in enumerate(class_names):
    print(f"{idx} -> {name}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


### What is happening in the code?

- We import all required libraries such as NumPy, Matplotlib, PIL, and TensorFlow/Keras.
- We define the training and testing folder paths.
- We create a function `load_images_from_folder()` that:
  - reads images from each class folder,
  - converts them to grayscale,
  - resizes them to 28×28,
  - normalizes pixel values,
  - and stores both image data and labels.
- Then we reshape the images into `(num_samples, 28, 28, 1)` because Keras expects a channel dimension.
- Next, we one-hot encode the labels using `to_categorical()`.
- Finally, we display a few sample images to verify the data loaded correctly.

#Task 2: Build the FCN Model
The worksheet asks for a Sequential model with 3 hidden layers of 64, 128, and 256 neurons, using sigmoid activation, and an output layer of 10 neurons with softmax.

# Task 2: Build the FCN Model

## Model Architecture

In this task, we create a Fully Connected Network (FCN) using Keras Sequential API.

Architecture:
- Input layer
- Flatten layer
- Dense layer with 64 neurons and sigmoid activation
- Dense layer with 128 neurons and sigmoid activation
- Dense layer with 256 neurons and sigmoid activation
- Output layer with 10 neurons and softmax activation

In [ ]:
# ============================
# Task 2: Build the FCN Model
# ============================

model = keras.Sequential([
    keras.layers.Input(shape=(28, 28, 1)),
    keras.layers.Flatten(),
    keras.layers.Dense(64, activation="sigmoid"),
    keras.layers.Dense(128, activation="sigmoid"),
    keras.layers.Dense(256, activation="sigmoid"),
    keras.layers.Dense(10, activation="softmax")
])

model.summary()

### What is happening in the code?

- `Input(shape=(28, 28, 1))` defines the shape of one grayscale image.
- `Flatten()` converts each 2D image into a 1D vector of size 784.
- The three `Dense()` layers are the hidden layers of the fully connected network.
- `sigmoid` is used as the activation function in all hidden layers as required in the worksheet.
- The final output layer has 10 neurons because there are 10 digit classes.
- `softmax` converts the final outputs into probabilities for each class.

#Task 3: Compile the Model
The worksheet suggests using an optimizer like Adam, a loss function like sparse categorical crossentropy, and metric accuracy. Since we already one-hot encoded labels above, the best matching loss is categorical_crossentropy. If you want to use sparse_categorical_crossentropy, then do not one-hot encode the labels.

# Task 3: Compile the Model

## Model Compilation

In this task, we compile the model by specifying:
- optimizer
- loss function
- evaluation metric

In [ ]:
# ============================
# Task 3: Compile the Model
# ============================

model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

print("Model compiled successfully.")

### What is happening in the code?

- `optimizer="adam"` tells Keras how to update weights during training.
- `loss="categorical_crossentropy"` measures the error between predicted probabilities and the true one-hot encoded labels.
- `metrics=["accuracy"]` tracks how many predictions are correct.
- Compilation prepares the model for training.

#Task 4: Train the Model
The worksheet asks for:

batch_size = 128
epochs = 20
validation_split = 0.2
optionally ModelCheckpoint and EarlyStopping

# Task 4: Train the Model

## Model Training

In this task, we train the FCN using:
- batch size = 128
- epochs = 20
- validation split = 0.2

We also use:
- ModelCheckpoint to save the best model
- EarlyStopping to stop training if validation loss stops improving

In [ ]:
# ============================
# Task 4: Train the Model
# ============================

batch_size = 128
epochs = 20

callbacks = [
    keras.callbacks.ModelCheckpoint(
        filepath="best_devnagari_fcn_model.h5",
        monitor="val_loss",
        save_best_only=True,
        verbose=1
    ),
    keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=4,
        restore_best_weights=True,
        verbose=1
    )
]

history = model.fit(
    x_train,
    y_train_cat,
    batch_size=batch_size,
    epochs=epochs,
    validation_split=0.2,
    callbacks=callbacks,
    verbose=1
)

### What is happening in the code?

- `batch_size=128` means 128 images are processed at one time before weights are updated.
- `epochs=20` means the full training data is passed through the network 20 times.
- `validation_split=0.2` uses 20% of the training data for validation.
- `ModelCheckpoint` saves the best model file during training.
- `EarlyStopping` stops training if the model stops improving on validation loss.
- `history` stores the training and validation loss/accuracy values for each epoch.

#Visualization of Training and Validation Performance
## Visualization

Now we plot:
- training loss
- validation loss
- training accuracy
- validation accuracy

In [ ]:
# -------------------------------------------------
# Plot training and validation loss and accuracy
# -------------------------------------------------

train_loss = history.history["loss"]
val_loss = history.history["val_loss"]
train_acc = history.history["accuracy"]
val_acc = history.history["val_accuracy"]

plt.figure(figsize=(12, 5))

# Plot loss
plt.subplot(1, 2, 1)
plt.plot(train_loss, label="Training Loss")
plt.plot(val_loss, label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training and Validation Loss")
plt.legend()

# Plot accuracy
plt.subplot(1, 2, 2)
plt.plot(train_acc, label="Training Accuracy")
plt.plot(val_acc, label="Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Training and Validation Accuracy")
plt.legend()

plt.tight_layout()
plt.show()

### What is happening in the code?

- `history.history` contains the values recorded during training.
- We extract loss and accuracy for both training and validation sets.
- The first graph helps us see whether the model is minimizing error.
- The second graph helps us see whether the model is learning better over time.
- If training accuracy is high but validation accuracy is low, the model may be overfitting.

#Task 5: Evaluate the Model


## Model Evaluation

After training, we evaluate the model on the test set to measure:
- test loss
- test accuracy

In [ ]:
# ============================
# Task 5: Evaluate the Model
# ============================

# Evaluate the trained model on test data
test_loss, test_accuracy = model.evaluate(x_test, y_test_cat, verbose=2)

print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f}")

### What is happening in the code?

- `model.evaluate()` checks how well the trained model performs on unseen test data.
- It returns:
  - loss: how much error the model makes,
  - accuracy: the fraction of correctly classified samples.
- Test accuracy is one of the final results required in the worksheet.

# Task 6: Save and Load the Model

## Model Saving and Loading

In this task, we:
- save the trained model to an `.h5` file
- load it again
- re-evaluate it on the test set

In [ ]:
# ============================
# Task 6: Save and Load the Model
# ============================

# Save the final trained model
model.save("devnagari_fcn_model.h5")
print("Model saved as devnagari_fcn_model.h5")

# Load the saved model
loaded_model = tf.keras.models.load_model("devnagari_fcn_model.h5")
print("Model loaded successfully.")

# Re-evaluate the loaded model
loaded_test_loss, loaded_test_accuracy = loaded_model.evaluate(x_test, y_test_cat, verbose=2)

print(f"Loaded Model Test Loss: {loaded_test_loss:.4f}")
print(f"Loaded Model Test Accuracy: {loaded_test_accuracy:.4f}")

### What is happening in the code?

- `model.save()` stores the model architecture, learned weights, and optimizer state.
- `load_model()` loads that saved model file back into memory.
- We evaluate the loaded model again to verify it performs the same as the original trained model.

# Task 7: Predictions

## Making Predictions

In this task, we:
- use the model to predict class probabilities on test images,
- convert probabilities into class labels using `np.argmax()`,
- and compare predicted labels with true labels.

In [ ]:
# ============================
# Task 7: Predictions
# ============================

# Predict class probabilities on the test set
predictions = loaded_model.predict(x_test)

# Convert probabilities to predicted class labels
predicted_labels = np.argmax(predictions, axis=1)

# True labels are currently one-hot encoded, so convert them back to integers
true_labels = np.argmax(y_test_cat, axis=1)

# Print some sample predictions
for i in range(min(10, len(x_test))):
    print(f"Image {i+1}: Predicted = {predicted_labels[i]}, True = {true_labels[i]}")

### What is happening in the code?

- `model.predict(x_test)` gives a probability distribution across all 10 classes for each test image.
- `np.argmax(predictions, axis=1)` selects the index with the highest probability, which becomes the predicted digit.
- Since the true labels were one-hot encoded, we convert them back using `np.argmax(y_test_cat, axis=1)`.
- Then we compare predicted and actual values.

**Visualizing Predictions**

In [ ]:
# -------------------------------------------------
# Display some test images with predicted and true labels
# -------------------------------------------------

plt.figure(figsize=(12, 6))

for i in range(min(10, len(x_test))):
    plt.subplot(2, 5, i + 1)
    plt.imshow(x_test[i].reshape(28, 28), cmap="gray")
    plt.title(f"P: {predicted_labels[i]}, T: {true_labels[i]}")
    plt.axis("off")

plt.suptitle("Sample Predictions on Test Images")
plt.tight_layout()
plt.show()

### What is happening in the code?

- We display a few test images.
- `P` means predicted label.
- `T` means true label.
- This helps visually inspect whether the model is classifying test digits correctly.

# Conclusion

In this worksheet, I successfully built a Fully Connected Network (FCN) for Devnagari digit classification using TensorFlow and Keras.

The workflow included:
- loading and preprocessing image data using PIL,
- building a Sequential FCN with three hidden layers,
- compiling the model with Adam and categorical crossentropy,
- training with validation split and callbacks,
- evaluating on test data,
- saving and loading the model,
- and making predictions on unseen images.

This exercise helped me understand the complete deep learning pipeline for image classification using Keras.